In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp

def load_30_industry_portfolios_daily(csv_path, start="20070701", end="20091231"):
    """
    Liest den ersten Tabellenblock ("Average Value Weighted Returns -- Daily")
    aus der Kenneth-French CSV und filtert auf [start, end] (Format YYYYMMDD).
    """
    with open(csv_path, "r") as f:
        lines = f.readlines()

    header_idx = next(i for i, l in enumerate(lines) if "Average Value Weighted Returns -- Daily" in l)
    colnames_idx = header_idx + 1
    while lines[colnames_idx].strip() == "":
        colnames_idx += 1
    colnames = [c.strip() for c in lines[colnames_idx].strip().split(",")]
    colnames[0] = "date"

    data_rows = []
    i = colnames_idx + 1
    while i < len(lines) and lines[i].strip() != "":
        row = [v.strip() for v in lines[i].strip().split(",")]
        data_rows.append(row)
        i += 1

    df = pd.DataFrame(data_rows, columns=colnames)
    df["date"] = df["date"].astype(int)
    for c in colnames[1:]:
        df[c] = df[c].astype(float)
    df = df.set_index("date")
    df = df.loc[(df.index >= int(start)) & (df.index <= int(end))]
    df = df.replace([-99.99, -999], np.nan)
    return df

csv_path = "/Users/SonjaGagel/Desktop/Masterarbeit/30_Industry_Portfolios_Daily.csv"
df_daily = load_30_industry_portfolios_daily(csv_path)
print("Shape:", df_daily.shape)
print("Zeitraum:", df_daily.index.min(), "-", df_daily.index.max())
df_daily.head()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/SonjaGagel/Desktop/Masterarbeit/30_Industry_Portfolios_Daily.csv'

In [3]:
from scipy.stats import jarque_bera, skew, kurtosis

def jb_test_summary(returns_df, alpha=0.05):
    """
    Fuehrt den Jarque-Bera-Test auf jede Spalte (Asset) eines Renditen-DataFrames aus.
    Analog zur Methodik von Huang et al. (2010), die den JB-Test zur Pruefung der
    Normalverteilungsannahme auf ihre Equity-Daten anwenden.
    """
    results = []
    for col in returns_df.columns:
        series = returns_df[col].dropna()
        stat, p_value = jarque_bera(series)
        results.append({
            "Asset": col,
            "Skewness": skew(series),
            "Kurtosis (Exzess)": kurtosis(series),  # kurtosis() liefert Exzess-Kurtosis (Normal = 0)
            "JB-Statistik": stat,
            "p-Wert": p_value,
            "H0 verworfen (5%)": p_value < alpha,
        })
    return pd.DataFrame(results).set_index("Asset")


jb_daily_crisis = jb_test_summary(df_daily)
print(f"Anteil Assets mit verworfener Normalverteilungsannahme (5%-Niveau): "
      f"{jb_daily_crisis['H0 verworfen (5%)'].mean():.1%}")
jb_daily_crisis.round(4)

Anteil Assets mit verworfener Normalverteilungsannahme (5%-Niveau): 100.0%


,Skewness,Kurtosis (Exzess),JB-Statistik,p-Wert,H0 verworfen (5%)
Asset,,,,,
Food,0.1560,5.8352,899.2144,0.0,True
Beer,0.5663,7.6538,1576.3896,0.0,True
Smoke,0.3358,8.9194,2106.8655,0.0,True
Games,0.0241,1.9178,96.9101,0.0,True
Books,0.1693,2.7175,197.4867,0.0,True
Hshld,0.0629,4.8452,618.6266,0.0,True
Clths,0.1592,2.3374,146.5351,0.0,True
Hlth,0.2612,8.1854,1771.5329,0.0,True
Chems,-0.2987,3.1591,272.1930,0.0,True


In [4]:
df_calm = load_30_industry_portfolios_daily(
    "30_Industry_Portfolios_Daily.csv", start="20040101", end="20061231"
)
print("Ruhige Periode Shape:", df_calm.shape)

jb_calm = jb_test_summary(df_calm)
print(f"Ruhige Periode (2004-2006): Anteil verworfen (5%): {jb_calm['H0 verworfen (5%)'].mean():.1%}")
print(f"Krisenperiode (2007-2009): Anteil verworfen (5%): {jb_daily_crisis['H0 verworfen (5%)'].mean():.1%}")
print()
print("Durchschnittliche Exzess-Kurtosis, ruhige Periode:", round(jb_calm["Kurtosis (Exzess)"].mean(), 3))
print("Durchschnittliche Exzess-Kurtosis, Krisenperiode:  ", round(jb_daily_crisis["Kurtosis (Exzess)"].mean(), 3))

Ruhige Periode Shape: (755, 30)
Ruhige Periode (2004-2006): Anteil verworfen (5%): 73.3%
Krisenperiode (2007-2009): Anteil verworfen (5%): 100.0%

Durchschnittliche Exzess-Kurtosis, ruhige Periode: 1.37
Durchschnittliche Exzess-Kurtosis, Krisenperiode:   4.304


In [5]:
def split_by_date_ranges(returns_df, boundaries):
    """
    Teilt eine Zeitreihe anhand vorgegebener Datumsgrenzen (YYYYMMDD) in Phasen auf.
    boundaries: Liste von (name, start, end) Tupeln.
    """
    blocks = {}
    for name, start, end in boundaries:
        mask = (returns_df.index >= start) & (returns_df.index <= end)
        blocks[name] = returns_df.loc[mask]
    return blocks


boundaries = [
    ("Vorkrise",  20070702, 20080914),
    ("Crash",     20080915, 20090309),
    ("Erholung",  20090310, 20091231),
]

crisis_blocks = split_by_date_ranges(df_daily, boundaries)

for name, block in crisis_blocks.items():
    print(f"{name:<10}: {block.index.min()} - {block.index.max()}  ({len(block)} Handelstage)")

Vorkrise  : 20070702 - 20080912  (304 Handelstage)
Crash     : 20080915 - 20090309  (121 Handelstage)
Erholung  : 20090310 - 20091231  (207 Handelstage)


In [6]:
S = len(df_daily)  # Gesamtzahl Szenarien (Handelstage)
dates_all = df_daily.index.to_numpy()

# Nominalverteilung: alle S Tage gleichgewichtet (Standardannahme)
pi_0 = np.full(S, 1.0 / S)

# Experten-Verteilungen: je nur die Tage der eigenen Phase gleichgewichtet, Rest = 0
pi_experts = {}
for name, block in crisis_blocks.items():
    pi_i = np.zeros(S)
    mask = np.isin(dates_all, block.index.to_numpy())
    pi_i[mask] = 1.0 / mask.sum()
    pi_experts[name] = pi_i
    print(f"{name}: Summe = {pi_i.sum():.4f}, Anzahl Tage mit pi>0 = {(pi_i > 0).sum()}")

print(f"\nNominalverteilung pi_0: Summe = {pi_0.sum():.4f}")

Vorkrise: Summe = 1.0000, Anzahl Tage mit pi>0 = 304
Crash: Summe = 1.0000, Anzahl Tage mit pi>0 = 121
Erholung: Summe = 1.0000, Anzahl Tage mit pi>0 = 207

Nominalverteilung pi_0: Summe = 1.0000


In [7]:
pi_matrix = np.stack(list(pi_experts.values()))  # Shape (m=3, S)

eta_lower = np.max(pi_0[None, :] - pi_matrix, axis=0)
eta_lower = np.clip(eta_lower, 0, None)

eta_upper = np.max(pi_matrix - pi_0[None, :], axis=0)
eta_upper = np.clip(eta_upper, 0, None)

print("eta_lower (eta): Min/Max =", eta_lower.min(), "/", eta_lower.max())
print("eta_upper (eta_bar): Min/Max =", eta_upper.min(), "/", eta_upper.max())
print()
print("Anzahl Szenarien mit eta_lower > 0:", (eta_lower > 0).sum(), "von", S)
print("Anzahl Szenarien mit eta_upper > 0:", (eta_upper > 0).sum(), "von", S)

# Konsistenzcheck: jede Expertenverteilung muss innerhalb der Box liegen
for name, pi_i in pi_experts.items():
    innerhalb = np.all((pi_0 - eta_lower - 1e-10 <= pi_i) & (pi_i <= pi_0 + eta_upper + 1e-10))
    print(f"{name} liegt vollstaendig in der Box: {innerhalb}")

eta_lower (eta): Min/Max = 0.0015822784810126582 / 0.0015822784810126582
eta_upper (eta_bar): Min/Max = 0.001707195203197868 / 0.006682184328904698

Anzahl Szenarien mit eta_lower > 0: 632 von 632
Anzahl Szenarien mit eta_upper > 0: 632 von 632
Vorkrise liegt vollstaendig in der Box: True
Crash liegt vollstaendig in der Box: True
Erholung liegt vollstaendig in der Box: True


In [8]:
R = df_daily.to_numpy()  # Shape (S, n) -- Renditematrix, Zeilen=Tage, Spalten=Assets
n = R.shape[1]

# Sinnvoller Zielrenditebereich fuer TAEGLICHE Renditen (viel kleiner als bei Monatsdaten!)
mean_returns_daily = R.T @ pi_0  # gewichteter Mittelwert je Asset unter pi_0
print("Durchschnittliche taegliche Rendite je Asset (Bereich):", 
      round(mean_returns_daily.min(), 4), "bis", round(mean_returns_daily.max(), 4))

Durchschnittliche taegliche Rendite je Asset (Bereich): -0.0803 bis 0.139


In [9]:
def solve_nominal_cvar_lp(R, pi, target_mu, epsilon=0.95, w0=1.0, x_lower=0.0, x_upper=1.0):
    """
    Nominales CVaR-Portfolio als LP (Rockafellar-Uryasev), vgl. Formel (3.18)
    mit eta=eta_bar=0 (Spezialfall ohne Box-Unsicherheit).
    R: (S,n) Renditematrix, pi: (S,) Wahrscheinlichkeitsvektor.
    """
    S, n = R.shape
    x = cp.Variable(n)
    gamma = cp.Variable()
    u = cp.Variable(S)

    objective = cp.Minimize(gamma + (1 / (1 - epsilon)) * (pi @ u))
    constraints = [
        cp.sum(x) == w0,
        x >= x_lower,
        x <= x_upper,
        x @ (R.T @ pi) >= target_mu,   # Renditebeschraenkung unter pi
        u >= -R @ x - gamma,
        u >= 0,
    ]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.ECOS)
    return x.value, prob.value, prob.status


# Test mit target_mu = 0.05 (0.05% taegliche Mindestrendite)
target_mu_daily = 0.05
x_nom, val_nom, status_nom = solve_nominal_cvar_lp(R, pi_0, target_mu_daily, epsilon=0.95)

print("Status:", status_nom)
print("Optimaler CVaR-Wert:", round(val_nom, 4))
print("Summe der Gewichte:", round(x_nom.sum(), 4))
print("Min/Max Gewicht:", round(x_nom.min(), 4), "/", round(x_nom.max(), 4))

NameError: name 'cp' is not defined